# Inferenzgraph- und Systembezugsanalyse

Dieses Notebook konstruiert aus den Ergebnissen von Stufe 1, Stufe 2 und der
Systembezugsklassifikation einen **gerichteten Inferenzgraphen** und analysiert
dessen Struktur im Hinblick auf inferentielle Relationen UND Systembezug.

**Analysen:**
1. Graphkonstruktion und Basismetriken
2. Tragende Knoten (implizite Hauptthesen)
3. Unbegründete Fundamentalbehauptungen
4. Relationstyp-Verteilung
5. Inferentielle Dichte pro Kapitel
6. Konnektoranalyse
7. Betweenness-Centrality
8. **Systembezug-Verteilung pro Kapitel**
9. **Systemgrenzen-überschreitende Kanten**
10. **Verknüpfungsmuster-Analyse (LIT-NLIT)**
11. Kapitelweise interaktive Visualisierung (mit Systembezug-Färbung)
12. Export

## 1. Setup

In [1]:
import re
import json
import pandas as pd
import numpy as np
import networkx as nx
from pyvis.network import Network
from collections import Counter, defaultdict
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [2]:
# ===== KONFIGURATION =====

STUFE1_JSON = "Beutin_complete_Direktzitat_stufe1.json"
STUFE2_JSON = "Beutin_complete_Direktzitat_stufe2_inter.json"  # None falls nicht vorhanden
SYSTEMBEZUG_CSV = "Beutin_complete_Direktzitat_systembezug.csv"

STEM = Path(STUFE1_JSON).stem.replace('_stufe1', '')
OUTPUT_METRICS = f"{STEM}_metriken.xlsx"


## 2. Daten laden und Graph konstruieren

In [3]:
with open(STUFE1_JSON, 'r', encoding='utf-8') as f:
    stufe1 = json.load(f)

stufe2 = []
if STUFE2_JSON and Path(STUFE2_JSON).exists():
    with open(STUFE2_JSON, 'r', encoding='utf-8') as f:
        stufe2 = json.load(f)
    print(f"Stufe 2 geladen: {len(stufe2)} Satzpaare")

# Systembezug laden
sys_lookup = {}  # pid -> {systembezug, lit_gegenstand, nlit_gegenstand}
df_sys = pd.DataFrame()  # bleibt leer, falls kein Systembezug geladen wird
if SYSTEMBEZUG_CSV and Path(SYSTEMBEZUG_CSV).exists():
    df_sys = pd.read_csv(SYSTEMBEZUG_CSV, encoding='utf-8-sig')
    for _, row in df_sys.iterrows():
        pid = row.get('Propositions-ID', '')
        if pd.notna(pid) and pid:
            sys_lookup[pid] = {
                'systembezug': str(row.get('Systembezug', '')).strip(),
                'lit_gegenstand': str(row.get('LIT-Gegenstand', '')).strip()
                    if pd.notna(row.get('LIT-Gegenstand')) else '',
                'nlit_gegenstand': str(row.get('NLIT-Gegenstand', '')).strip()
                    if pd.notna(row.get('NLIT-Gegenstand')) else '',
            }
    print(f"Systembezug geladen: {len(sys_lookup)} Propositionen")
    sys_counts = Counter(v['systembezug'] for v in sys_lookup.values())
    for s, c in sys_counts.most_common():
        print(f"  {s}: {c}")
else:
    print("Kein Systembezug geladen (SYSTEMBEZUG_CSV nicht gefunden)")

# Propositions-Lookup
prop_lookup = {}
for item in stufe1:
    sid = item.get('satz_id', '')
    kapitel = item.get('kapitel', '(Ohne Kapitel)')
    for p in item.get('propositionen', []):
        pid = p.get('id', p.get('prop_id', ''))
        sys_info = sys_lookup.get(pid, {})
        prop_lookup[pid] = {
            'text': p.get('text', ''),
            'satz_id': sid,
            'kapitel': kapitel,
            'relationstyp': p.get('relationstyp', 'PR-UNA'),
            'systembezug': sys_info.get('systembezug', ''),
            'lit_gegenstand': sys_info.get('lit_gegenstand', ''),
            'nlit_gegenstand': sys_info.get('nlit_gegenstand', ''),
        }

print(f"Propositionen: {len(prop_lookup)}")


Stufe 2 geladen: 10853 Satzpaare
Systembezug geladen: 29446 Propositionen
  LIT: 16888
  LIT-NLIT: 7572
  NLIT: 4986
Propositionen: 29446


In [4]:
def normalize_rel(r):
    von = r.get('von', r.get('from', r.get('source', '')))
    nach = r.get('nach', r.get('to', r.get('target', '')))
    typ = r.get('typ', r.get('type', r.get('relationstyp', '')))
    kon = r.get('konnektor', r.get('connector', r.get('marker', '')))
    return von, nach, typ, kon

edges = []
for item in stufe1:
    for r in item.get('intra_relationen', item.get('relationen', [])):
        von, nach, typ, kon = normalize_rel(r)
        if von and nach and typ:
            edges.append({'von': von, 'nach': nach, 'typ': typ,
                          'konnektor': kon, 'scope': 'intra'})
for item in stufe2:
    for r in item.get('inter_relationen', []):
        von, nach, typ, kon = normalize_rel(r)
        if von and nach and typ:
            edges.append({'von': von, 'nach': nach, 'typ': typ,
                          'konnektor': kon, 'scope': 'inter'})

print(f"Kanten gesamt: {len(edges)}")
print(f"  Intra: {sum(1 for e in edges if e['scope']=='intra')}")
print(f"  Inter: {sum(1 for e in edges if e['scope']=='inter')}")


Kanten gesamt: 15624
  Intra: 9810
  Inter: 5814


In [5]:
# Globaler Graph
G = nx.DiGraph()
for pid, info in prop_lookup.items():
    G.add_node(pid, text=info['text'][:80], satz_id=info['satz_id'],
               kapitel=info['kapitel'], full_text=info['text'],
               relationstyp=info['relationstyp'],
               systembezug=info['systembezug'])
for e in edges:
    G.add_edge(e['von'], e['nach'], typ=e['typ'],
               konnektor=e['konnektor'], scope=e['scope'])

print(f"Globaler Graph: {G.number_of_nodes()} Knoten, {G.number_of_edges()} Kanten")
print(f"Schwach zusammenhängende Komponenten: {nx.number_weakly_connected_components(G)}")
isolated = sum(1 for pid in G.nodes() if G.degree(pid) == 0)
print(f"Isolierte Knoten (PR-UNA): {isolated}")

# Kapitelweise Subgraphen
chapter_graphs = {}
chapters_list = list(dict.fromkeys(info['kapitel'] for info in prop_lookup.values()))
for ch in chapters_list:
    ch_nodes = [pid for pid, info in prop_lookup.items() if info['kapitel'] == ch]
    ch_G = G.subgraph(ch_nodes).copy()
    chapter_graphs[ch] = ch_G
    print(f"  {ch[:45]}: {ch_G.number_of_nodes()} Knoten, {ch_G.number_of_edges()} Kanten")


Globaler Graph: 29446 Knoten, 15593 Kanten
Schwach zusammenhängende Komponenten: 14048
Isolierte Knoten (PR-UNA): 8240
  Mittelalterliche Literatur: 2089 Knoten, 1067 Kanten
  (Unbekannt): 9 Knoten, 2 Kanten
  Humanismus und Reformation: 1856 Knoten, 950 Kanten
  Literatur des Barock: 1587 Knoten, 882 Kanten
  Aufklärung: 1255 Knoten, 847 Kanten
  Kunstepoche: 2131 Knoten, 1240 Kanten
  Vormärz: 1794 Knoten, 953 Kanten
  Realismus und Gründerzeit: 1883 Knoten, 1012 Kanten
  Die literarische Moderne (1890–1920): 1691 Knoten, 900 Kanten
  Literatur in der Weimarer Republik: 1765 Knoten, 973 Kanten
  Literatur im Dritten Reich: 634 Knoten, 360 Kanten
  Die deutsche Literatur des Exils: 919 Knoten, 491 Kanten
  Deutsche Literatur nach 1945: 1130 Knoten, 620 Kanten
  Die Literatur der DDR: 2699 Knoten, 1451 Kanten
  Die Literatur der Bundesrepublik: 3002 Knoten, 1611 Kanten
  Tendenzen in der deutschsprachigen Gegenwarts: 5002 Knoten, 2234 Kanten


## 3. Tragende Knoten (global)

In [6]:
in_deg = sorted(G.in_degree(), key=lambda x: -x[1])

print("=== Tragende Knoten (h\u00f6chster In-Degree) ===")
rows_in = []
for pid, deg in in_deg[:15]:
    if deg > 0:
        info = prop_lookup.get(pid, {})
        incoming = Counter(G.edges[u, pid].get('typ', '') for u in G.predecessors(pid))
        types_str = ', '.join(f"{t}:{c}" for t, c in incoming.most_common())
        print(f"  {pid} (In: {deg}) [{info.get('systembezug','')}, {info.get('kapitel','')[:25]}]")
        print(f"    {info.get('text', '')[:100]}")
        print(f"    Typen: {types_str}\n")
        rows_in.append({'Propositions-ID': pid, 'In-Degree': deg,
                        'Kapitel': info.get('kapitel',''),
                        'Systembezug': info.get('systembezug',''),
                        'Proposition': info.get('text','')[:120],
                        'Eingehende Typen': types_str})


=== Tragende Knoten (höchster In-Degree) ===
  P(8531.1) (In: 9) [LIT, Die Literatur der Bundesr]
    Die Beobachtung Baumgarts bezog sich auf ein literarhistorisch bemerkenswertes Phänomen.
    Typen: PR-BER:9

  P(106.1) (In: 6) [LIT-NLIT, Mittelalterliche Literatu]
    Während der Zeit der Völkerwanderung entstanden mehrere Sagenkreise.
    Typen: PR-FLG:5, PR-BER:1

  P(1742.1) (In: 6) [LIT, Literatur des Barock]
    Theater im Deutschland des 17. Jahrhunderts bedeutet vielerlei.
    Typen: PR-BER:6

  P(3812.1) (In: 6) [LIT, Vormärz]
    In den sich breit entwickelnden, spezifisch journalistischen Genres dominierten die Autoren des Jung
    Typen: PR-BER:6

  P(7797.3) (In: 6) [LIT, Die Literatur der DDR]
    Die Ignorierung des zweiten Buchmarkts durch die großen Institutionen der Literaturvermittlung gesch
    Typen: PR-BER:6

  P(8190.1) (In: 6) [NLIT, Die Literatur der Bundesr]
    Selbstzweifel innerhalb der jungen Generation, unter den Intellektuellen und unter den Arbeitern

## 4. Unbegr\u00fcndete Fundamentalbehauptungen

In [7]:
fundamentals = []
for pid in G.nodes():
    in_d, out_d = G.in_degree(pid), G.out_degree(pid)
    if out_d > 0 and in_d == 0:
        info = prop_lookup.get(pid, {})
        fundamentals.append({
            'pid': pid, 'out_degree': out_d,
            'kapitel': info.get('kapitel', ''),
            'systembezug': info.get('systembezug', ''),
            'text': info.get('text', ''),
        })
fundamentals.sort(key=lambda x: -x['out_degree'])

print("=== Unbegr\u00fcndete Fundamentalbehauptungen ===")
for f in fundamentals[:15]:
    print(f"  {f['pid']} (Out: {f['out_degree']}) [{f['systembezug']}, {f['kapitel'][:25]}]")
    print(f"    {f['text'][:100]}\n")


=== Unbegründete Fundamentalbehauptungen ===
  P(417.1) (Out: 4) [LIT, Mittelalterliche Literatu]
    Parzifal entreißt Jeschute, der Gattin des Herzogs Orilus, in Unkenntnis der Bedeutung des Minnepfan

  P(1107.2) (Out: 4) [NLIT, Humanismus und Reformatio]
    Luther gab jeweils einer größeren Versammlung die Macht zu entscheiden.

  P(1696.1) (Out: 4) [LIT-NLIT, Literatur des Barock]
    Er sieht sich als den schon von Böhme erwarteten Jüngling.

  P(1705.2) (Out: 4) [NLIT, Literatur des Barock]
    Die idealisierte Vergangenheit ist eine statische, hierarchisch gegliederte Welt.

  P(3758.1) (Out: 4) [LIT, Vormärz]
    Im Vormärz bestand keine scharfe Trennung zwischen ›hoher‹ und ›niedriger‹ Literatur, sondern ein fl

  P(5622.9) (Out: 4) [LIT, Literatur in der Weimarer]
    Der Expressionismus zeigte sich ganz außerstande, die Welt als Objekt menschlicher Praxis zu erkläre

  P(5767.3) (Out: 4) [LIT-NLIT, Literatur in der Weimarer]
    Den meisten Autoren fehlte die »Phantasie fü

## 5. Relationstyp-Verteilung

In [8]:
type_counts = Counter(e['typ'] for e in edges)
scope_type = defaultdict(Counter)
for e in edges:
    scope_type[e['scope']][e['typ']] += 1

total = len(edges)
print(f"{'Typ':<10} {'Gesamt':>7} {'Intra':>7} {'Inter':>7} {'Anteil':>8}")
print('-' * 45)
for typ, count in type_counts.most_common():
    print(f"{typ:<10} {count:>7} {scope_type['intra'][typ]:>7} {scope_type['inter'][typ]:>7} {count/total*100:>7.1f}%")


Typ         Gesamt   Intra   Inter   Anteil
---------------------------------------------
PR-BER       14183    8503    5680    90.8%
PR-INK         907     863      44     5.8%
PR-FLG         523     433      90     3.3%
PR-UNA          11      11       0     0.1%


## 6. Inferentielle Dichte pro Kapitel

In [9]:
density_rows = []
for ch, ch_G in chapter_graphs.items():
    n_nodes = ch_G.number_of_nodes()
    n_edges = ch_G.number_of_edges()
    isolated = sum(1 for pid in ch_G.nodes() if ch_G.degree(pid) == 0)
    density_rows.append({
        'Kapitel': ch,
        'Propositionen': n_nodes,
        'Relationen': n_edges,
        'Isolierte (PR-UNA)': isolated,
        'Vernetzungsgrad': round((n_nodes - isolated) / max(n_nodes, 1), 2),
        # vormals faelschlich "Dichte": Kanten pro Knoten = halber mittlerer Grad
        'Relationen pro Proposition': round(n_edges / max(n_nodes, 1), 2),
        # korrekt definierte Dichte des gerichteten Graphen: m / (n*(n-1))
        'Graphdichte (gerichtet)': round(nx.density(ch_G), 6),
        'Komponenten': nx.number_weakly_connected_components(ch_G),
        'Größte Komponente': max((len(c) for c in nx.weakly_connected_components(ch_G)), default=0),
    })
df_density = pd.DataFrame(density_rows)
print("=== Inferentielle Vernetzung pro Kapitel ===")
print("Hinweis: 'Relationen pro Proposition' ist KEINE Graphdichte;")
print("die echte Dichte steht in 'Graphdichte (gerichtet)'.")
print(df_density.to_string(index=False))


=== Inferentielle Vernetzung pro Kapitel ===
Hinweis: 'Relationen pro Proposition' ist KEINE Graphdichte;
die echte Dichte steht in 'Graphdichte (gerichtet)'.
                                                         Kapitel  Propositionen  Relationen  Isolierte (PR-UNA)  Vernetzungsgrad  Relationen pro Proposition  Graphdichte (gerichtet)  Komponenten  Größte Komponente
                                      Mittelalterliche Literatur           2089        1067                 583             0.72                        0.51                 0.000245         1032                 15
                                                     (Unbekannt)              9           2                   5             0.44                        0.22                 0.027778            7                  2
                                      Humanismus und Reformation           1856         950                 541             0.71                        0.51                 0.000276          922     

## 7. Konnektoranalyse

In [10]:
EXPLIZIT = {'weil', 'denn', 'daher', 'deshalb', 'daraus', 'folglich',
            'sondern', 'aber', 'jedoch', 'dennoch', 'obwohl', 'wenngleich',
            'insbesondere', 'zumindest', 'wegen'}

def classify_konnektor(kon):
    if not kon: return 'kein Konnektor'
    k = kon.lower().strip()
    if 'implizit' in k: return 'implizit'
    if any(e in k for e in EXPLIZIT): return 'explizit-argumentativ'
    if 'relativsatz' in k or 'apposition' in k: return 'syntaktisch'
    if k in ('und', ',', 'sowie'): return 'koordinierend'
    return 'sonstig'

kon_classes = Counter(classify_konnektor(e['konnektor']) for e in edges)
total_k = sum(kon_classes.values())
print("=== Konnektorklassen ===")
for cls, count in kon_classes.most_common():
    print(f"  {cls:<25} {count:>4} ({count/total_k*100:.1f}%)")


=== Konnektorklassen ===
  implizit                  8583 (54.9%)
  sonstig                   4566 (29.2%)
  explizit-argumentativ     1112 (7.1%)
  syntaktisch                822 (5.3%)
  koordinierend              536 (3.4%)
  kein Konnektor               5 (0.0%)


## 8. Betweenness-Centrality

In [11]:
# --- Betweenness in drei Varianten -----------------------------------------
# (1) bc_raw  : rohe Pfadzaehlung. Absolute Zahl kuerzester Pfade durch den
#               Knoten. NUR innerhalb einer Komponente / eines Kapitels als
#               Rangordnung interpretierbar.
# (2) bc_norm : komponentennormalisiert, bc_raw / ((k-1)(k-2)) mit k = Groesse
#               der schwachen Komponente. Wertebereich [0,1], kapitelueber-
#               greifend skalengleich -- ABER: beguenstigt Kleinstkomponenten
#               (Mittelknoten einer 3er-Kette = 0.5). Nicht allein ranken!
# (3) xbc     : Cross-System-Betweenness. Anzahl kuerzester Pfade zwischen
#               LIT- und NLIT-Knoten (beide Richtungen) durch den Knoten.
#               Absolute Vermittlungslast zwischen den Systemen; DIE Groesse
#               fuer das kapiteluebergreifende Bruecken-Ranking.

bc_raw, bc_norm, xbc, comp_size = {}, {}, {}, {}

for comp in nx.weakly_connected_components(G):
    k = len(comp)
    for v in comp:
        comp_size[v] = k
    if k < 3:
        continue
    sub = G.subgraph(comp)

    raw = nx.betweenness_centrality(sub, normalized=False)
    bc_raw.update(raw)
    denom = (k - 1) * (k - 2)
    bc_norm.update({v: val / denom for v, val in raw.items()})

    lit  = [v for v in comp if prop_lookup.get(v, {}).get('systembezug') == 'LIT']
    nlit = [v for v in comp if prop_lookup.get(v, {}).get('systembezug') == 'NLIT']
    if lit and nlit:
        b1 = nx.betweenness_centrality_subset(sub, sources=lit,  targets=nlit, normalized=False)
        b2 = nx.betweenness_centrality_subset(sub, sources=nlit, targets=lit,  normalized=False)
        for v in comp:
            val = b1.get(v, 0) + b2.get(v, 0)
            if val > 0:
                xbc[v] = val

for pid in G.nodes():
    bc_raw.setdefault(pid, 0.0)
    bc_norm.setdefault(pid, 0.0)
    xbc.setdefault(pid, 0.0)
    comp_size.setdefault(pid, 1)

# --- Bruecken-Score: verbindet der Knoten real beide Systeme? ---------------
# min(#LIT-Nachbarn, #NLIT-Nachbarn) ueber ein- und ausgehende Kanten.
bridge_score = {}
for pid in G.nodes():
    nb = set(G.predecessors(pid)) | set(G.successors(pid))
    nb.discard(pid)  # Selbstschleifen nicht als Nachbarschaft zaehlen
    n_lit  = sum(1 for u in nb if prop_lookup.get(u, {}).get('systembezug') == 'LIT')
    n_nlit = sum(1 for u in nb if prop_lookup.get(u, {}).get('systembezug') == 'NLIT')
    bridge_score[pid] = min(n_lit, n_nlit)

# Abwaertskompatibilitaet: 'bc' bleibt die Rohgroesse (Zelle "Export" nutzt sie)
bc = bc_raw

print("=== Argumentative Knotenpunkte ===")
print(f"Knoten mit Roh-Betweenness > 0:   {sum(1 for v in bc_raw.values() if v > 0)}")
print(f"Knoten mit Cross-System-BC > 0:   {sum(1 for v in xbc.values() if v > 0)}")
print(f"Knoten mit Bruecken-Score > 0:    {sum(1 for v in bridge_score.values() if v > 0)}")
print("\nTop 10 nach Cross-System-Betweenness:")
for pid, score in sorted(xbc.items(), key=lambda x: -x[1])[:10]:
    if score > 0:
        info = prop_lookup.get(pid, {})
        print(f"  {pid} (XBC: {score:.1f}, roh: {bc_raw[pid]:.1f}, Komp: {comp_size[pid]}) "
              f"[{info.get('systembezug','')}, {info.get('kapitel','')[:25]}]")
        print(f"    {info.get('text','')[:100]}\n")


=== Argumentative Knotenpunkte ===
Knoten mit Roh-Betweenness > 0:   4211
Knoten mit Cross-System-BC > 0:   467
Knoten mit Bruecken-Score > 0:    453

Top 10 nach Cross-System-Betweenness:
  P(7078.1) (XBC: 9.0, roh: 12.0, Komp: 19) [LIT-NLIT, Die Literatur der DDR]
    Das konnte gerade in der Literatur nicht zu einem weniger kritischen Verhältnis des DDR-Bewohners zu

  P(3380.2) (XBC: 8.0, roh: 8.0, Komp: 19) [LIT-NLIT, Vormärz]
    Die Literatur des Vormärz gibt in durchaus ambivalenter Weise Auskunft über einen fundamentalen Umst

  P(7079.3) (XBC: 8.0, roh: 16.0, Komp: 19) [LIT, Die Literatur der DDR]
    Die DDR-Literatur der 60er Jahre ist geprägt von einem Anwachsen kritischer Tendenzen.

  P(3381.1) (XBC: 6.0, roh: 12.0, Komp: 19) [NLIT, Vormärz]
    Die durch die industrielle Revolution beschleunigte bürgerlich-kapitalistische Veränderung der ökono

  P(5342.1) (XBC: 6.0, roh: 6.0, Komp: 6) [LIT-NLIT, Literatur in der Weimarer]
    Nach 1934 war Kalékos Karriere in Deutschla

## 9. Systembezug-Verteilung pro Kapitel

Wie verteilen sich LIT, NLIT und LIT-NLIT über die Kapitel?
Der LIT-NLIT-Anteil zeigt die Verknüpfungsdichte der Literaturgeschichte.

In [12]:
if sys_lookup:
    sys_rows = []
    for ch in chapters_list:
        ch_props = [pid for pid, info in prop_lookup.items() if info['kapitel'] == ch]
        n_total = len(ch_props)
        n_lit = sum(1 for pid in ch_props if prop_lookup[pid]['systembezug'] == 'LIT')
        n_nlit = sum(1 for pid in ch_props if prop_lookup[pid]['systembezug'] == 'NLIT')
        n_litnlit = sum(1 for pid in ch_props if prop_lookup[pid]['systembezug'] == 'LIT-NLIT')
        n_unknown = n_total - n_lit - n_nlit - n_litnlit
        sys_rows.append({
            'Kapitel': ch, 'Propositionen': n_total,
            'LIT': n_lit, 'NLIT': n_nlit, 'LIT-NLIT': n_litnlit,
            'Ohne': n_unknown,
            'LIT %': round(n_lit / max(n_total, 1) * 100, 1),
            'NLIT %': round(n_nlit / max(n_total, 1) * 100, 1),
            'LIT-NLIT %': round(n_litnlit / max(n_total, 1) * 100, 1),
        })
    df_sys = pd.DataFrame(sys_rows)
    print("=== Systembezug pro Kapitel ===")
    print(df_sys.to_string(index=False))
else:
    print("Systembezug-Daten nicht verfügbar.")
    df_sys = pd.DataFrame()


=== Systembezug pro Kapitel ===
                                                         Kapitel  Propositionen  LIT  NLIT  LIT-NLIT  Ohne  LIT %  NLIT %  LIT-NLIT %
                                      Mittelalterliche Literatur           2089 1190   533       366     0   57.0    25.5        17.5
                                                     (Unbekannt)              9    5     4         0     0   55.6    44.4         0.0
                                      Humanismus und Reformation           1856  804   630       422     0   43.3    33.9        22.7
                                            Literatur des Barock           1587 1068   277       242     0   67.3    17.5        15.2
                                                      Aufklärung           1255  768   145       342     0   61.2    11.6        27.3
                                                     Kunstepoche           2131 1335   280       516     0   62.6    13.1        24.2
                              

## 10. Systemgrenzen-überschreitende Kanten

Welche inferentiellen Relationen verlaufen *innerhalb* eines Systems
(LIT\u2192LIT, NLIT\u2192NLIT) und welche *überschreiten* die Systemgrenze
(LIT\u2192NLIT, NLIT\u2192LIT, LIT-NLIT beteiligt)?
Dies ist die zentrale Metrik für das Verknüpfungsproblem.

In [13]:
if sys_lookup:
    edge_sys_types = Counter()
    cross_edges = []
    
    for e in edges:
        sys_von = prop_lookup.get(e['von'], {}).get('systembezug', '')
        sys_nach = prop_lookup.get(e['nach'], {}).get('systembezug', '')
        
        if not sys_von or not sys_nach:
            edge_sys_types['(unklassifiziert)'] += 1
            continue
        
        pair = f"{sys_von} \u2192 {sys_nach}"
        edge_sys_types[pair] += 1
        
        # Systemgrenzüberschreitung: verschiedene Systeme oder LIT-NLIT beteiligt
        is_cross = (sys_von != sys_nach) or 'LIT-NLIT' in (sys_von, sys_nach)
        if is_cross:
            cross_edges.append({
                'von': e['von'], 'nach': e['nach'], 'typ': e['typ'],
                'sys_von': sys_von, 'sys_nach': sys_nach,
                'text_von': prop_lookup.get(e['von'], {}).get('text', '')[:60],
                'text_nach': prop_lookup.get(e['nach'], {}).get('text', '')[:60],
            })
    
    total_classified = sum(c for k, c in edge_sys_types.items() if k != '(unklassifiziert)')
    print("=== Inferentielle Relationen nach Systembezug ===")
    print(f"{'Paar':<25} {'Anzahl':>7} {'Anteil':>8}")
    print('-' * 45)
    for pair, count in edge_sys_types.most_common():
        pct = count / max(total_classified, 1) * 100
        print(f"{pair:<25} {count:>7} {pct:>7.1f}%")
    
    n_within = sum(c for p, c in edge_sys_types.items()
                   if p in ('LIT \u2192 LIT', 'NLIT \u2192 NLIT'))
    n_cross = sum(c for p, c in edge_sys_types.items()
                  if p not in ('LIT \u2192 LIT', 'NLIT \u2192 NLIT', '(unklassifiziert)'))
    print(f"\nInnerhalb eines Systems: {n_within} ({n_within/max(total_classified,1)*100:.1f}%)")
    print(f"Systemgrenzüberschreitend: {n_cross} ({n_cross/max(total_classified,1)*100:.1f}%)")
    
    if cross_edges:
        print(f"\nBeispiele systemübergreifender Kanten:")
        for ce in cross_edges[:10]:
            print(f"  {ce['von']} [{ce['sys_von']}] \u2192 {ce['nach']} [{ce['sys_nach']}]: {ce['typ']}")
            print(f"    {ce['text_von']}")
            print(f"    \u2192 {ce['text_nach']}")
else:
    print("Systembezug-Daten nicht verfügbar.")


=== Inferentielle Relationen nach Systembezug ===
Paar                       Anzahl   Anteil
---------------------------------------------
LIT → LIT                    6989    44.7%
LIT-NLIT → LIT-NLIT          2529    16.2%
NLIT → NLIT                  1903    12.2%
LIT-NLIT → LIT               1574    10.1%
LIT → LIT-NLIT               1160     7.4%
NLIT → LIT-NLIT               717     4.6%
LIT-NLIT → NLIT               366     2.3%
NLIT → LIT                    267     1.7%
LIT → NLIT                    119     0.8%

Innerhalb eines Systems: 8892 (56.9%)
Systemgrenzüberschreitend: 6732 (43.1%)

Beispiele systemübergreifender Kanten:
  P(001.2) [LIT-NLIT] → P(001.1) [NLIT]: PR-BER
    Die Vorbemerkungen sollen verhindern, dass der Eindruck erwe
    → Begriffliche, historische, geographische und politische Vorb
  P(020.6) [LIT-NLIT] → P(020.7) [NLIT]: PR-BER
    Die Besteigung des Mont Ventoux durch Francesco Petrarca am 
    → Die Besteigung des Mont Ventoux am 26. April 1336 ist ei

## 11. Verknüpfungsmuster-Analyse (LIT-NLIT)

Analyse der LIT-NLIT-Propositionen: Welche literarischen und nicht-literarischen
Gegenstände werden verknüpft? Welche Muster dominieren?

In [14]:
if sys_lookup:
    litnlit = [(pid, info) for pid, info in prop_lookup.items()
               if info['systembezug'] == 'LIT-NLIT']
    
    print(f"=== LIT-NLIT Verknüpfungsstellen: {len(litnlit)} ===")
    
    if litnlit:
        # Alle Verknüpfungspaare auflisten
        print(f"\n{'PID':<12} {'LIT-Gegenstand':<30} {'NLIT-Gegenstand':<30} {'Kapitel':<25}")
        print('-' * 100)
        for pid, info in litnlit:
            lit_g = info.get('lit_gegenstand', '')[:29]
            nlit_g = info.get('nlit_gegenstand', '')[:29]
            ch = info.get('kapitel', '')[:24]
            print(f"{pid:<12} {lit_g:<30} {nlit_g:<30} {ch:<25}")
        
        # Häufigste LIT-Gegenstände
        lit_counter = Counter(info['lit_gegenstand'] for _, info in litnlit
                              if info['lit_gegenstand'])
        nlit_counter = Counter(info['nlit_gegenstand'] for _, info in litnlit
                               if info['nlit_gegenstand'])
        
        if lit_counter:
            print(f"\nHäufigste LIT-Gegenstände:")
            for g, c in lit_counter.most_common(10):
                print(f"  {g}: {c}")
        if nlit_counter:
            print(f"\nHäufigste NLIT-Gegenstände:")
            for g, c in nlit_counter.most_common(10):
                print(f"  {g}: {c}")
        
        # Verknüpfungen pro Kapitel
        ch_litnlit = Counter(info['kapitel'] for _, info in litnlit)
        print(f"\nLIT-NLIT pro Kapitel:")
        for ch, c in ch_litnlit.most_common():
            total_ch = sum(1 for pid, info in prop_lookup.items() if info['kapitel'] == ch)
            print(f"  {ch[:45]}: {c} ({c/max(total_ch,1)*100:.1f}%)")
        
        # Inferentielle Einbettung der LIT-NLIT-Propositionen
        litnlit_pids = {pid for pid, _ in litnlit}
        connected = sum(1 for pid in litnlit_pids if G.degree(pid) > 0)
        isolated_ln = len(litnlit_pids) - connected
        print(f"\nInferentielle Einbettung der LIT-NLIT-Propositionen:")
        print(f"  Vernetzt (mind. 1 Relation): {connected} ({connected/max(len(litnlit),1)*100:.0f}%)")
        print(f"  Isoliert (PR-UNA): {isolated_ln} ({isolated_ln/max(len(litnlit),1)*100:.0f}%)")
else:
    print("Systembezug-Daten nicht verfügbar.")


=== LIT-NLIT Verknüpfungsstellen: 7572 ===

PID          LIT-Gegenstand                 NLIT-Gegenstand                Kapitel                  
----------------------------------------------------------------------------------------------------
P(001.2)     Geschichte der deutschsprachi  universale Weltgeschichte      Mittelalterliche Literat 
P(001.3)     Geschichte der deutschsprachi  universale Weltgeschichte      Mittelalterliche Literat 
P(009.1)     Literatur, Philosophie und Sp  niederländische Humanisten     Mittelalterliche Literat 
P(020.6)     Francesco Petrarca (Dichter)   Befreiung von höfischer Etike  Mittelalterliche Literat 
P(020.8)     dramatisches Werk Shakespeare  Aufbruch des modernen Gewisse  Mittelalterliche Literat 
P(020.9)     dramatisches Werk Shakespeare  Übergang aus feudalistischen   Mittelalterliche Literat 
P(022.9)     Witz, Anekdotik und Novellist  lebhaft-beredter Umgang mitei  Mittelalterliche Literat 
P(025.3)     Fränkisch-Alemannische als Li  Frä

## 12. Kapitelweise interaktive Visualisierung

Knotenfarbe nach Systembezug:
- **Blau**: LIT (literarisches System)
- **Orange**: NLIT (nicht-literarisches System)
- **Rot/Violett**: LIT-NLIT (Verknüpfungsstellen)
- **Grau**: Kein Systembezug / PR-UNA isoliert

Kantenfarbe nach Relationstyp. Gestrichelt = Inter-Satz.

In [15]:
EDGE_COLORS = {
    'PR-FLG': '#2196F3',
    'PR-BER': '#4CAF50',
    'PR-INK': '#F44336',
}

SYS_COLORS = {
    'LIT': '#42A5F5',       # Blau
    'NLIT': '#FF9800',      # Orange
    'LIT-NLIT': '#AB47BC',  # Violett
    '': '#BDBDBD',          # Grau (kein Systembezug)
}


def create_chapter_network(ch_name, ch_G, output_path):
    if ch_G.number_of_nodes() == 0:
        print(f"  {ch_name}: Keine Knoten, übersprungen.")
        return
    
    net = Network(height='800px', width='100%', directed=True,
                  notebook=True, cdn_resources='remote')
    net.barnes_hut(gravity=-3000, central_gravity=0.3,
                   spring_length=150, spring_strength=0.01)
    
    max_in = max((d for _, d in ch_G.in_degree()), default=1)
    
    for pid in ch_G.nodes():
        info = prop_lookup.get(pid, {})
        in_d = ch_G.in_degree(pid)
        out_d = ch_G.out_degree(pid)
        is_isolated = ch_G.degree(pid) == 0
        sys_bez = info.get('systembezug', '')
        size = 8 if is_isolated else 10 + (in_d / max(max_in, 1)) * 30
        
        title = (f"<b>{pid}</b> [{info.get('satz_id','')}]<br>"
                 f"Systembezug: {sys_bez}<br>"
                 f"{info.get('text','')}<br><br>"
                 f"In: {in_d} | Out: {out_d}")
        
        if sys_bez == 'LIT-NLIT':
            lit_g = info.get('lit_gegenstand', '')
            nlit_g = info.get('nlit_gegenstand', '')
            if lit_g or nlit_g:
                title += f"<br>LIT: {lit_g} \u2194 NLIT: {nlit_g}"
        
        color = SYS_COLORS.get(sys_bez, '#BDBDBD')
        if is_isolated and not sys_bez:
            color = '#E0E0E0'
        
        label = f"{pid}\n{info.get('text','')[:40]}..."
        net.add_node(pid, label=label, title=title, size=size,
                     color=color, font={'size': 8})
    
    for von, nach, data in ch_G.edges(data=True):
        typ = data.get('typ', '')
        scope = data.get('scope', '')
        color = EDGE_COLORS.get(typ, '#999999')
        width = 3 if scope == 'inter' else 1.5
        dashes = scope == 'inter'
        title = f"{von} \u2192 {nach}: {typ} | {data.get('konnektor','')}"
        net.add_edge(von, nach, title=title, color=color,
                     width=width, dashes=dashes, arrows='to')
    
    net.show(output_path)
    n_lit = sum(1 for pid in ch_G.nodes() if prop_lookup.get(pid,{}).get('systembezug')=='LIT')
    n_nlit = sum(1 for pid in ch_G.nodes() if prop_lookup.get(pid,{}).get('systembezug')=='NLIT')
    n_ln = sum(1 for pid in ch_G.nodes() if prop_lookup.get(pid,{}).get('systembezug')=='LIT-NLIT')
    print(f"  {ch_name[:40]}: {ch_G.number_of_nodes()} Kn, {ch_G.number_of_edges()} Ka "
          f"[LIT:{n_lit} NLIT:{n_nlit} LIT-NLIT:{n_ln}] \u2192 {output_path}")


In [16]:
print("=== Kapitelweise Visualisierung ===")
print("Knoten: Blau=LIT, Orange=NLIT, Violett=LIT-NLIT, Grau=ohne")
print("Kanten: Grün=PR-BER, Blau=PR-FLG, Rot=PR-INK, gestrichelt=Inter-Satz\n")

for ch in chapters_list:
    safe_name = re.sub(r'[^a-zA-Z0-9_]', '_', ch)[:40]
    output_path = f"{STEM}_graph_{safe_name}.html"
    create_chapter_network(ch, chapter_graphs[ch], output_path)


=== Kapitelweise Visualisierung ===
Knoten: Blau=LIT, Orange=NLIT, Violett=LIT-NLIT, Grau=ohne
Kanten: Grün=PR-BER, Blau=PR-FLG, Rot=PR-INK, gestrichelt=Inter-Satz

Beutin_complete_Direktzitat_graph_Mittelalterliche_Literatur.html
  Mittelalterliche Literatur: 2089 Kn, 1067 Ka [LIT:1190 NLIT:533 LIT-NLIT:366] → Beutin_complete_Direktzitat_graph_Mittelalterliche_Literatur.html
Beutin_complete_Direktzitat_graph__Unbekannt_.html
  (Unbekannt): 9 Kn, 2 Ka [LIT:5 NLIT:4 LIT-NLIT:0] → Beutin_complete_Direktzitat_graph__Unbekannt_.html
Beutin_complete_Direktzitat_graph_Humanismus_und_Reformation.html
  Humanismus und Reformation: 1856 Kn, 950 Ka [LIT:804 NLIT:630 LIT-NLIT:422] → Beutin_complete_Direktzitat_graph_Humanismus_und_Reformation.html
Beutin_complete_Direktzitat_graph_Literatur_des_Barock.html
  Literatur des Barock: 1587 Kn, 882 Ka [LIT:1068 NLIT:277 LIT-NLIT:242] → Beutin_complete_Direktzitat_graph_Literatur_des_Barock.html
Beutin_complete_Direktzitat_graph_Aufkl_rung.html
  Aufklä

## 13. Export: Metriken als Excel

In [17]:
from openpyxl.styles import Font, PatternFill, Alignment

# Sheet 1: Knotenmetriken
node_rows = []
for pid in G.nodes():
    info = prop_lookup.get(pid, {})
    node_rows.append({
        'Propositions-ID': pid,
        'Kapitel': info.get('kapitel', ''),
        'Satz-ID': info.get('satz_id', ''),
        'Proposition': info.get('text', ''),
        'Systembezug': info.get('systembezug', ''),
        'LIT-Gegenstand': info.get('lit_gegenstand', ''),
        'NLIT-Gegenstand': info.get('nlit_gegenstand', ''),
        'In-Degree': G.in_degree(pid),
        'Out-Degree': G.out_degree(pid),
        'Betweenness (roh)': round(bc_raw.get(pid, 0), 5),
        'Betweenness (komponentennormalisiert)': round(bc_norm.get(pid, 0), 6),
        'Cross-System-Betweenness': round(xbc.get(pid, 0), 5),
        'Brücken-Score': bridge_score.get(pid, 0),
        'Komponentengröße': comp_size.get(pid, 1),
        'Relationstyp': info.get('relationstyp', ''),
    })
df_nodes = pd.DataFrame(node_rows).sort_values('In-Degree', ascending=False)

# Sheet 2: Kantenliste
df_edges = pd.DataFrame(edges)

# Sheet 3: Typverteilung
df_types = pd.DataFrame([
    {'Typ': t, 'Anzahl': c, 'Anteil': f"{c/len(edges)*100:.1f}%",
     'Intra': scope_type['intra'][t], 'Inter': scope_type['inter'][t]}
    for t, c in type_counts.most_common()
])

with pd.ExcelWriter(OUTPUT_METRICS, engine='openpyxl') as writer:
    df_nodes.to_excel(writer, index=False, sheet_name='Knotenmetriken')
    df_edges.to_excel(writer, index=False, sheet_name='Kantenliste')
    df_density.to_excel(writer, index=False, sheet_name='Dichte pro Kapitel')
    df_types.to_excel(writer, index=False, sheet_name='Typverteilung')
    if len(df_sys) > 0:
        df_sys.to_excel(writer, index=False, sheet_name='Systembezug pro Kapitel')
    
    hf = PatternFill(start_color='2B4C7E', end_color='2B4C7E', fill_type='solid')
    for ws in writer.sheets.values():
        for cell in ws[1]:
            cell.fill = hf
            cell.font = Font(name='Arial', size=10, bold=True, color='FFFFFF')
            cell.alignment = Alignment(wrap_text=True, vertical='top')
        ws.auto_filter.ref = ws.dimensions

print(f"Metriken exportiert: {OUTPUT_METRICS}")


Metriken exportiert: Beutin_complete_Direktzitat_metriken.xlsx


## 14. Zusammenfassung

**Inferentielle Analysen (Ebene 0):**
1. Tragende Knoten (In-Degree): Implizite Hauptthesen
2. Fundamentalbehauptungen (Out-Degree ohne In-Degree): Unbegründete Axiome
3. Knotenpunkte (Betweenness): Argumentative Schaltstellen
4. Relationstyp-Verteilung: PR-BER vs. PR-FLG vs. PR-INK
5. Konnektorklassen: Explizierungsgrad
6. Inferentielle Dichte und Vernetzungsgrad pro Kapitel

**Systembezug-Analysen (Ebene 1+2):**
7. Systembezug-Verteilung pro Kapitel (LIT/NLIT/LIT-NLIT-Anteile)
8. Systemgrenzen-überschreitende Kanten: Wie viele Relationen verlaufen
   zwischen verschiedenen Systemen?
9. Verknüpfungsmuster: Welche literarischen und nicht-literarischen
   Gegenstände werden in LIT-NLIT-Propositionen verknüpft?

**Visuelle Kodierung:**
- Knotenfarbe: Blau (LIT), Orange (NLIT), Violett (LIT-NLIT), Grau (ohne)
- Kantenfarbe: Grün (PR-BER), Blau (PR-FLG), Rot (PR-INK)
- Gestrichelte Kanten: Inter-Satz-Relationen